In [24]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, r2_score


In [19]:
housing = fetch_california_housing()
help(housing)

Help on Bunch in module sklearn.utils._bunch object:

class Bunch(builtins.dict)
 |  Bunch(**kwargs)
 |  
 |  Container object exposing keys as attributes.
 |  
 |  Bunch objects are sometimes used as an output for functions and methods.
 |  They extend dictionaries by enabling values to be accessed by key,
 |  `bunch["value_key"]`, or by an attribute, `bunch.value_key`.
 |  
 |  Examples
 |  --------
 |  >>> from sklearn.utils import Bunch
 |  >>> b = Bunch(a=1, b=2)
 |  >>> b['b']
 |  2
 |  >>> b.b
 |  2
 |  >>> b.a = 3
 |  >>> b['a']
 |  3
 |  >>> b.c = 6
 |  >>> b['c']
 |  6
 |  
 |  Method resolution order:
 |      Bunch
 |      builtins.dict
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __dir__(self)
 |      Default dir() implementation.
 |  
 |  __getattr__(self, key)
 |  
 |  __init__(self, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  __setattr__(self, key, value)
 |      Implement setattr(self, name, value).
 |  
 

In [20]:
X = housing.data
y = housing.target

df = pd.DataFrame(X, columns=housing.feature_names)
df['TARGET'] = y

df.head()


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,TARGET
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [21]:
 # Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [22]:
# Initialize the scaler
scaler = StandardScaler()

# Fit and transform the training data, and transform the test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [23]:
# Initialize PCA, specifying the number of components
pca = PCA(n_components=0.95)  # Keep 95% of the variance

# Fit PCA on the scaled training data and transform both the training and test data
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Number of components selected
print(f"Number of components selected: {pca.n_components_}")


Number of components selected: 6


In [25]:
# Initialize the model (Ridge Regression in this case)
model = ElasticNet()

# Define the parameter grid for Grid Search
param_grid = {
    'alpha': [0.1, 1, 10, 100],  # Regularization strength
}

# Initialize Grid Search with cross-validation
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error')

# Fit Grid Search on the training data
grid_search.fit(X_train_pca, y_train)

# Best parameters found by Grid Search
print(f"Best parameters: {grid_search.best_params_}")

# Best model
best_model = grid_search.best_estimator_


Best parameters: {'alpha': 0.1}


In [28]:
rmse = (-grid_search.best_score_)**0.5
rmse

0.8247947668226658

In [32]:
# Evaluate the best model with cross-validation
cv_scores = cross_val_score(best_model, X_train_pca, y_train, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)

# Display cross-validation results
print(f"Cross-Validation RMSE: {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}")


Cross-Validation RMSE: 0.8248 ± 0.0073


In [31]:
# Predict on the test set
y_pred = best_model.predict(X_test_pca)

# Calculate and print the evaluation metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test R^2: {r2:.4f}")


Test RMSE: 0.8299
Test R^2: 0.4745


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_boston
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

# Load sample dataset
data = load_boston()
X, y = data.data, data.target

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define pipeline with preprocessing, scaling, PCA, and regression model
pipeline = Pipeline([
    ('scaler', StandardScaler()),            # Feature Scaling
    ('poly_features', PolynomialFeatures()), # Polynomial Features
    ('pca', PCA()),                          # Principal Component Analysis
    ('regressor', Ridge())                   # Regression Model (Ridge Regression)
])

# Define parameter grid for Grid Search
param_grid = {
    'poly_features__degree': [1, 2, 3],          # Degrees of polynomial features
    'pca__n_components': [5, 10, 15],            # Number of PCA components
    'regressor__alpha': [0.01, 0.1, 1, 10, 100]  # Ridge regularization parameter
}

# Grid Search with Cross-Validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Fit the model
grid_search.fit(X_train, y_train)

# Best parameters and score
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Score:", np.sqrt(-grid_search.best_score_))

# Evaluate on test data
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2 = r2_score(y_test, y_pred)

print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test R2 Score: {test_r2:.4f}")
